# NVIDIA OVRTX — Minimal Example (Colab T4)

> `robot-ovrtx.usda` 씬을 로드하여 PNG로 렌더링 후 원본 이미지와 비교합니다.

**실행 전 필수:** 런타임 → 런타임 유형 변경 → **T4 GPU** (또는 L4) 선택

---

## Step 1 — GPU 확인

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "❌ nvidia-smi 실패 — 런타임 유형을 T4 GPU로 변경하세요!")

## Step 2 — 환경 설정 (Vulkan ICD + ovrtx 설치)

In [ ]:
import subprocess, sys, json, os, time
from pathlib import Path

# ── 1. 시스템 라이브러리 ─────────────────────────────────────────
print("[1/5] Installing system libs...")
subprocess.run(["apt-get", "install", "-y", "-q",
                "libvulkan1", "libvulkan-dev", "vulkan-tools",
                "libegl1", "xvfb",
                "libx11-6", "libx11-xcb1", "libxcb1",
                "libxext6", "libxau6", "libxdmcp6"],
               check=True, capture_output=True)
print("  OK")

# ── 2. NVIDIA 라이브러리 진단 ─────────────────────────────────────
print("[2/5] Diagnosing NVIDIA Vulkan...")

# Pre-installed ICD JSON 읽기 (덮어쓰기 전에 확인)
pre_icd = Path("/usr/share/vulkan/icd.d/nvidia_icd.json")
orig_lib_path = None
if pre_icd.exists():
    try:
        orig_lib_path = json.loads(pre_icd.read_text())["ICD"]["library_path"]
        print(f"  Pre-installed ICD library_path: {orig_lib_path}")
    except Exception as e:
        print(f"  Pre-installed ICD parse error: {e}")
else:
    print("  No pre-installed nvidia_icd.json")

# libGLX_nvidia.so.0 탐색
r = subprocess.run('find /usr /lib -name "libGLX_nvidia.so.0" 2>/dev/null',
                   shell=True, capture_output=True, text=True)
all_glx = [p.strip() for p in r.stdout.splitlines() if p.strip()]
print(f"  Found libGLX copies: {all_glx}")

nvidia_lib     = all_glx[0] if all_glx else "/usr/lib64-nvidia/libGLX_nvidia.so.0"
nvidia_lib_dir = str(Path(nvidia_lib).parent)

# nm -D 타입 확인: T/W = export(ICD), U = import(ICD 아님)
for lib in all_glx:
    nm_r = subprocess.run(
        f'nm -D "{lib}" 2>/dev/null | grep vk_icdGetInstanceProcAddr',
        shell=True, capture_output=True, text=True)
    line = nm_r.stdout.strip()
    sym_type = line.split()[1] if line and len(line.split()) >= 2 else "?"
    print(f"  nm -D {Path(lib).name}: '{line}' → type={sym_type}")
    if sym_type in ("T", "W", "D"):
        print(f"    ✅ EXPORTED — this IS the Vulkan ICD")
    elif sym_type == "U":
        print(f"    ⚠️ UNDEFINED/IMPORTED — not the ICD, just uses it")

# ldd
ldd = subprocess.run(["ldd", nvidia_lib], capture_output=True, text=True)
missing = [l.strip() for l in ldd.stdout.splitlines() if "not found" in l]
print("  ldd: ✅ OK" if not missing else f"  ldd: ⚠️ Missing: {missing}")

# ── TEST A: 기존 ICD + LD_LIBRARY_PATH ──────────────────────────
print("\n  [TEST A] Pre-installed ICD + LD_LIBRARY_PATH, no display...")
env_base = {**os.environ,
            "LD_LIBRARY_PATH": nvidia_lib_dir + ":" + os.environ.get("LD_LIBRARY_PATH", ""),
            "VK_LOADER_DEBUG": "error",
            "DISPLAY": ""}
vk_a = subprocess.run(["vulkaninfo", "--summary"], capture_output=True, text=True, env=env_base)
test_a = "NVIDIA" in vk_a.stdout or "Tesla" in vk_a.stdout
for line in vk_a.stdout.splitlines():
    if any(k in line for k in ("deviceName", "driverVersion")):
        print(f"    {line.strip()}")
print(f"    {'✅ PASS' if test_a else '✗ FAIL: ' + vk_a.stderr[:150]}")

# ── TEST B: Xvfb 가상 디스플레이 ────────────────────────────────
test_b = False
xvfb_proc = None
if not test_a:
    print("\n  [TEST B] Starting Xvfb :99 virtual display...")
    xvfb_proc = subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1280x720x24", "-ac"],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    env_b = {**env_base, "DISPLAY": ":99"}
    vk_b = subprocess.run(["vulkaninfo", "--summary"], capture_output=True, text=True, env=env_b)
    test_b = "NVIDIA" in vk_b.stdout or "Tesla" in vk_b.stdout
    for line in vk_b.stdout.splitlines():
        if any(k in line for k in ("deviceName", "driverVersion")):
            print(f"    {line.strip()}")
    print(f"    {'✅ PASS — Xvfb works!' if test_b else '✗ FAIL: ' + vk_b.stderr[:200]}")
    if not test_b and xvfb_proc:
        xvfb_proc.terminate()

display_val = ":99" if test_b else ""

# ── 3. ldconfig 등록 ─────────────────────────────────────────────
print("\n[3/5] Registering library path...")
Path("/etc/ld.so.conf.d/nvidia-ovrtx.conf").write_text(nvidia_lib_dir + "\n")
subprocess.run(["ldconfig"], check=False, capture_output=True)
print(f"  Registered: {nvidia_lib_dir}")

# ── 4. ICD — 경쟁 ICD만 제거, nvidia_icd.json은 원본 유지 ────────
print("[4/5] Cleaning competing ICDs (keeping NVIDIA)...")
icd_dir = Path("/usr/share/vulkan/icd.d")
icd_dir.mkdir(parents=True, exist_ok=True)
for f in icd_dir.glob("*.json"):
    if f.name != "nvidia_icd.json":
        try:
            content = f.read_text()
            if any(k in content.lower() for k in ("llvm", "mesa", "radeon", "intel", "lvp")):
                f.unlink()
                print(f"  Removed: {f.name}")
        except Exception:
            pass

remaining = [f.name for f in icd_dir.glob("*.json")]
print(f"  Remaining ICDs: {remaining}")

# Step 3 에서 읽을 환경 저장
Path("/tmp/ovrtx_env.json").write_text(json.dumps({
    "nvidia_lib_dir": nvidia_lib_dir,
    "display": display_val,
    "test_a": test_a,
    "test_b": test_b,
}))

# ── 5. ovrtx 설치 ────────────────────────────────────────────────
print("[5/5] Installing ovrtx...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ovrtx==0.3.0.312915", "numpy==2.2.6", "pillow==12.1.1"],
                   capture_output=True, text=True)
if r.returncode != 0:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "ovrtx==0.3.0.312915", "numpy==2.2.6", "pillow==12.1.1",
                        "--extra-index-url", "https://pypi.ngc.nvidia.com"],
                       capture_output=True, text=True)
print("  ✅ Done" if r.returncode == 0 else f"  ❌ {r.stderr[:200]}")
print(f"\n✅ Step 2 complete  |  display='{display_val}'  |  Vulkan={'OK' if (test_a or test_b) else 'FAIL'}")

## Step 3 — 렌더링 (subprocess · 환경변수 격리)

> `VK_ICD_FILENAMES`는 프로세스 시작 시 단 한 번 읽힙니다.  
> **subprocess로 새 Python 프로세스**를 생성해 렌더링합니다.

In [ ]:
import subprocess, sys, os, json, threading, time
from pathlib import Path

# Step 2 에서 저장한 환경 정보 읽기
_cfg = json.loads(Path("/tmp/ovrtx_env.json").read_text())
NVIDIA_LIB_DIR = _cfg["nvidia_lib_dir"]
DISPLAY_VAL    = _cfg["display"]

USD_URL    = "https://omniverse-content-production.s3.us-west-2.amazonaws.com/Samples/Robot-OVRTX/robot-ovrtx.usda"
OUTPUT_DIR = Path("_output")
OUTPUT_DIR.mkdir(exist_ok=True)
WARMUP     = 30

render_env = {
    **os.environ,
    "VK_ICD_FILENAMES": "/usr/share/vulkan/icd.d/nvidia_icd.json",
    "LD_LIBRARY_PATH" : NVIDIA_LIB_DIR + ":" + os.environ.get("LD_LIBRARY_PATH", ""),
    "VK_LOADER_DEBUG" : "error",
    "DISPLAY"         : DISPLAY_VAL,
}

render_script = f"""
import sys, warnings
warnings.filterwarnings("ignore")
import ovrtx, numpy as np
from PIL import Image
from pathlib import Path

USD_URL    = "{USD_URL}"
OUTPUT_DIR = Path("{OUTPUT_DIR}")
WARMUP     = {WARMUP}

print("ovrtx:", ovrtx.__version__, flush=True)
print("Creating renderer...", flush=True)
renderer = ovrtx.Renderer()
print("✅ Renderer created", flush=True)

print("Loading USD (네트워크 다운로드 — 수 분 소요)...", flush=True)
renderer.open_usd(USD_URL)
print("✅ USD loaded", flush=True)

print(f"Warming up ({{WARMUP}} frames)...", flush=True)
for i in range(WARMUP - 1):
    renderer.step(render_products={{"/Render/Camera"}}, delta_time=1/60)
    if (i + 1) % 10 == 0:
        print(f"  {{i+1}}/{{WARMUP}}", flush=True)

print("Capturing final frame...", flush=True)
products = renderer.step(render_products={{"/Render/Camera"}}, delta_time=1/60)

for pname, product in products.items():
    for idx, frame in enumerate(product.frames):
        var = frame.render_vars["LdrColor"].map(device=ovrtx.Device.CPU)
        pixels = np.from_dlpack(var)
        out = OUTPUT_DIR / f"render_{{idx}}.png"
        Image.fromarray(pixels).save(out)
        print(f"✅ Saved: {{out}}  ({{pixels.shape[1]}}x{{pixels.shape[0]}})", flush=True)
"""

# ── 스트리밍 subprocess (실시간 출력 + 30분 타임아웃) ──────────────
print(f"Starting render subprocess  DISPLAY='{DISPLAY_VAL}'")
print()

proc = subprocess.Popen(
    [sys.executable, "-u", "-c", render_script],  # -u: unbuffered
    env=render_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

stderr_lines = []
def _read_stderr():
    for line in proc.stderr:
        stderr_lines.append(line)
threading.Thread(target=_read_stderr, daemon=True).start()

# stdout 실시간 출력
start_t = time.time()
TIMEOUT = 1800  # 30분
timed_out = False
for line in proc.stdout:
    print(line, end="", flush=True)
    if time.time() - start_t > TIMEOUT:
        proc.kill()
        timed_out = True
        print(f"\n⏰ Killed after {TIMEOUT}s timeout")
        break

try:
    proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    proc.kill()

elapsed = int(time.time() - start_t)
print(f"\n--- subprocess exited: code={proc.returncode}  elapsed={elapsed}s ---")

if proc.returncode != 0 or timed_out:
    stderr_tail = "".join(stderr_lines)[-1500:]
    if stderr_tail.strip():
        print("─── STDERR ───")
        print(stderr_tail)
    if timed_out:
        raise RuntimeError("Render timed out (30 min). Check STDERR for last status.")
    raise RuntimeError(f"Render subprocess exited with code {proc.returncode}")

## Step 4 — 결과 표시 및 원본 비교

In [ ]:
import urllib.request, io
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

output_dir = Path("_output")
render_pngs = sorted(output_dir.glob("render_*.png"))

if not render_pngs:
    print("❌ 렌더 결과 없음 — Step 3을 먼저 실행하세요")
else:
    render_img = Image.open(render_pngs[0]).convert("RGB")
    print(f"렌더 결과: {render_pngs[0]}  ({render_img.width}x{render_img.height})")

    # Try multiple candidate URLs for the reference image
    REF_URLS = [
        "https://github.com/00saridon/PhysicalAI/raw/main/dashboard/public/example-minimal.jpg",
        "https://raw.githubusercontent.com/00saridon/PhysicalAI/main/dashboard/public/example-minimal.jpg",
    ]
    ref_img = None
    for url in REF_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=15) as resp:
                ref_img = Image.open(io.BytesIO(resp.read())).convert("RGB")
            print(f"Reference loaded from: {url}")
            break
        except Exception as e:
            print(f"⚠️ {url}: {e}")

    has_ref = ref_img is not None
    n = 2 if has_ref else 1
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 5), facecolor="#0a0c10")
    if n == 1:
        axes = [axes]

    axes[0].imshow(render_img)
    axes[0].set_title("OVRTX Render — Colab T4", color="#76b900",
                      fontsize=12, fontweight="bold", pad=8)
    axes[0].axis("off")

    if has_ref:
        axes[1].imshow(ref_img.resize(render_img.size, Image.LANCZOS))
        axes[1].set_title("Reference (example-minimal.jpg)", color="#94a3b8",
                          fontsize=12, fontweight="bold", pad=8)
        axes[1].axis("off")

    plt.tight_layout(pad=1.5)
    plt.savefig(output_dir / "comparison.png", dpi=150,
                facecolor="#0a0c10", bbox_inches="tight")
    plt.show()
    print("✅ comparison.png 저장 완료")

## Step 5 — PNG 다운로드

In [ ]:
from google.colab import files
from pathlib import Path

for png in sorted(Path("_output").glob("*.png")):
    files.download(str(png))
    print(f"Downloading: {png}")